# NB-03 — Métricas de avaliação (Cap. 2 §2.5 e Cap. 3)

Usa `lib/avaliacao.py`, que só chama `core/` e `backend/semantic.py` (nada reimplementado). Saídas em `saidas/NB-03/`.

BERTScore e juiz LLM dependem do ambiente de **produção** (node BERTScore, Ollama). Ligue `USAR_SEMANTICAS` e exporte as variáveis (`BERTSCORE_URL`, `NODE_TOKEN`, `JUDGE_MODEL`, `OLLAMA_HOST`, `OLLAMA_API_KEY`) onde o notebook rodar; sem isso, voltam como indisponíveis sem derrubar o resto.

In [1]:
# Parâmetros
from pathlib import Path
SAIDA = Path('saidas/NB-03')
SEED = 0
USAR_SEMANTICAS = True  # True só onde o ambiente de produção estiver acessível

In [2]:
import sys, json, platform
sys.path.insert(0, 'lib')
import pandas as pd
from avaliacao import avaliar, contagem_sid
SAIDA.mkdir(parents=True, exist_ok=True)

## 1. Exemplo trabalhado (Cap. 2 §2.5)
Contagem **à mão**: gabarito com N = 7 palavras; hipótese com 1 substituição (rato→rata), 1 remoção (a) e 1 inserção (velho). WER = (S+D+I)/N = 3/7 ≈ 0,4286. Em caracteres (26 no gabarito): 1 + 2 (`a␣`) + 6 (`␣velho`) = 9 → CER = 9/26 ≈ 0,3462.

In [3]:
REF = 'o rato roeu a roupa do rei'
HIP = 'o rata roeu roupa do rei velho'
sid = contagem_sid(HIP, REF)
r = avaliar(HIP, REF)
print('S/I/D/N código:', sid, '| à mão: S=1 I=1 D=1 N=7')
print(f"WER {r['wer']:.4f} (3/7 = {3/7:.4f})  CER {r['cer']:.4f} (9/26 = {9/26:.4f})  cosseno {r['cosseno']:.4f}")
assert sid == {'S':1,'I':1,'D':1,'N':7}

S/I/D/N código: {'S': 1, 'I': 1, 'D': 1, 'N': 7} | à mão: S=1 I=1 D=1 N=7
WER 0.4286 (3/7 = 0.4286)  CER 0.3462 (9/26 = 0.3462)  cosseno 0.7143


## 2. Quando o cosseno engana (Cap. 3, limitação metodológica)
O cosseno usa sacola de palavras sem ordem, acentos ou pontuação como informação.

In [4]:
casos = [
  ('Idêntico',               REF, REF),
  ('Ordem das palavras trocada', 'rei do roupa a roeu rato o', REF),
  ('Sem acentuação',         'a não é nao', 'a não é não'),
  ('Acentuação perdida',     'ele nao esta aqui', 'ele não está aqui'),
  ('Maiúscula e pontuação',  'O rato roeu.', 'o rato roeu'),
  ('Palavra trocada',        'o gato roeu a roupa do rei', REF),
]
linhas = []
for nome, hip, ref in casos:
    m = avaliar(hip, ref)
    linhas.append({'caso': nome, 'hipotese': hip, 'referencia': ref,
                   'cer': round(m['cer'],3), 'wer': round(m['wer'],3), 'cosseno': round(m['cosseno'],3)})
tab = pd.DataFrame(linhas)
tab.to_csv(SAIDA/'tabela_limitacao_cosseno.csv', index=False)
tab[['caso','cer','wer','cosseno']]

,caso,cer,wer,cosseno
0,Idêntico,0.000,0.000,1.000
1,Ordem das palavras trocada,0.615,0.857,1.000
2,Sem acentuação,0.091,0.250,0.816
3,Acentuação perdida,0.118,0.500,0.500
4,Maiúscula e pontuação,0.182,0.667,1.000
5,Palavra trocada,0.038,0.143,0.857


In [5]:
def tex(df):
    esc = lambda s: str(s).replace('_', r'\_').replace('%', r'\%')
    out = [r'\begin{tabular}{lrrr}', r'\hline', r'Caso & CER & WER & Cosseno \\', r'\hline']
    out += [f"{esc(r.caso)} & {r.cer:.3f} & {r.wer:.3f} & {r.cosseno:.3f} \\\\" for r in df.itertuples()]
    return '\n'.join(out + [r'\hline', r'\end{tabular}'])
(SAIDA/'tabela_limitacao_cosseno.tex').write_text(tex(tab))
print((SAIDA/'tabela_limitacao_cosseno.tex').read_text())

\begin{tabular}{lrrr}
\hline
Caso & CER & WER & Cosseno \\
\hline
Idêntico & 0.000 & 0.000 & 1.000 \\
Ordem das palavras trocada & 0.615 & 0.857 & 1.000 \\
Sem acentuação & 0.091 & 0.250 & 0.816 \\
Acentuação perdida & 0.118 & 0.500 & 0.500 \\
Maiúscula e pontuação & 0.182 & 0.667 & 1.000 \\
Palavra trocada & 0.038 & 0.143 & 0.857 \\
\hline
\end{tabular}


## 3. BERTScore e juiz (somente com o ambiente de produção)
Registra modelo/camada do BERTScore (como diz o texto: `bert-base-multilingual-cased`, camada 9), `JUDGE_MODEL` e a versão do Ollama (pendência `[Informar]` do fim do Cap. 2).

In [6]:
import os, requests
from core import bertscore as B
config = {'bertscore_modelo': B.MODEL, 'bertscore_camada': B.NUM_LAYERS,
          'judge_model': os.environ.get('JUDGE_MODEL'), 'ollama_versao': None,
          'bertscore_url_configurada': bool(os.environ.get('BERTSCORE_URL'))}
if USAR_SEMANTICAS:
    host = os.environ.get('OLLAMA_HOST', '').rstrip('/')
    key = os.environ.get('OLLAMA_API_KEY')
    try:
        h = {'Authorization': f'Bearer {key}'} if key else {}
        config['ollama_versao'] = requests.get(f'{host}/api/version', headers=h, timeout=10).json().get('version')
    except Exception as e:
        config['ollama_versao'] = f'indisponível ({type(e).__name__})'
    amostra = avaliar(HIP, REF, bertscore=True, juiz=True)
    config['amostra'] = {k: amostra[k] for k in ('bertscore', 'juiz', 'erros')}
config

{'bertscore_modelo': 'bert-base-multilingual-cased',
 'bertscore_camada': 9,
 'judge_model': 'qwen2.5:7b-instruct',
 'ollama_versao': '0.34.4',
 'bertscore_url_configurada': True,
 'amostra': {'bertscore': {'precision': 0.8912271857261658,
   'recall': 0.8781874179840088,
   'f1': 0.8846591711044312,
   'model': 'bert-base-multilingual-cased'},
  'juiz': {'fidelity': 3,
   'completeness': 4,
   'correctness': 2,
   'overall': 3.0,
   'comment': "Palavra 'rato' foi transcrita como 'rata', e 'rei' foi adicionado como 'rei velho', alterando o sentido.",
   'model': 'qwen2.5:7b-instruct'},
  'erros': {}}}

## Manifest

In [7]:
import subprocess
def git(*a):
    try: return subprocess.check_output(['git', *a], cwd='..', text=True).strip()
    except Exception: return None
manifest = {'notebook': 'NB-03', 'commit': git('rev-parse', '--short', 'HEAD'),
            'arvore_suja': bool(git('status', '--porcelain')), 'python': platform.python_version(),
            'usar_semanticas': USAR_SEMANTICAS, 'config': config}
(SAIDA/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
manifest

{'notebook': 'NB-03',
 'commit': '531bbad',
 'arvore_suja': True,
 'python': '3.11.2',
 'usar_semanticas': True,
 'config': {'bertscore_modelo': 'bert-base-multilingual-cased',
  'bertscore_camada': 9,
  'judge_model': 'qwen2.5:7b-instruct',
  'ollama_versao': '0.34.4',
  'bertscore_url_configurada': True,
  'amostra': {'bertscore': {'precision': 0.8912271857261658,
    'recall': 0.8781874179840088,
    'f1': 0.8846591711044312,
    'model': 'bert-base-multilingual-cased'},
   'juiz': {'fidelity': 3,
    'completeness': 4,
    'correctness': 2,
    'overall': 3.0,
    'comment': "Palavra 'rato' foi transcrita como 'rata', e 'rei' foi adicionado como 'rei velho', alterando o sentido.",
    'model': 'qwen2.5:7b-instruct'},
   'erros': {}}}}